# ONNX & TensorRT 部署 Demo

本项目演示 PyTorch 模型的两种主流部署方式：
- **ONNX Runtime**：跨平台、跨框架的通用推理方案
- **TensorRT**：NVIDIA GPU 上的极致推理加速方案

每种方式都包含 **Python 推理** 和 **C++ 推理** 的完整示例。

---
## 整体流程

```text
PyTorch 模型
    │
    ├─── torch.onnx.export() ──→ model.onnx ──┬──→ ONNX Runtime 推理（Python / C++）
    │                                          │
    │                                          └──→ trtexec ──→ model.engine ──→ TensorRT 推理（Python / C++）
    │
    └─── 共享同一个 SimpleMLP 模型（input=4, hidden=16, output=3）
```

---
## 项目结构

```text
ONNX_TensorRT_demo/
│
├── model/                          # ===== 模型定义 =====
│   ├── model.py                    # 共享模型定义（SimpleMLP）+ 保存权重
│   └── weights.pth                 # 模型权重文件（运行后生成）
│
├── onnx/                           # ===== ONNX 部署 =====
│   ├── export_onnx.py              # 导出 ONNX + 结构检查 + 数值精度对比
│   ├── infer_python.py             # ONNX Runtime Python 推理
│   ├── model.onnx                  # 导出的 ONNX 模型（运行后生成）
│   └── cpp/
│       ├── CMakeLists.txt          # CMake 构建配置（指向 onnxruntime/）
│       ├── main.cpp                # ONNX Runtime C++ 推理
│       └── build/                  # 编译输出（自动创建）
│
├── tensorrt/                       # ===== TensorRT 部署 =====
│   ├── build_engine.py             # 用 trtexec 构建 TensorRT Engine
│   ├── infer_python.py             # TensorRT Python 推理（pycuda）
│   ├── model.engine                # FP32 Engine（运行后生成）
│   ├── model_fp16.engine           # FP16 Engine（运行后生成）
│   └── cpp/
│       ├── CMakeLists.txt          # CMake 构建配置（指向 TensorRT/）
│       ├── main.cpp                # TensorRT C++ 推理
│       └── build/                  # 编译输出（自动创建）
│
├── onnxruntime/                    # ===== ONNX Runtime C++ 官方库（需下载）=====
│   │                               #   下载: onnxruntime-linux-x64-1.19.2.tgz
│   │                               #   解压后重命名为 onnxruntime/ 放于此
│   ├── include/                    #   C++ 头文件（onnxruntime_cxx_api.h 等）
│   └── lib/                        #   动态库（libonnxruntime.so 等）
│
├── TensorRT/                       # ===== TensorRT C++ 官方库（需下载）=====
│   │                               #   从 NVIDIA Developer 下载 tar 包
│   │                               #   解压后重命名为 TensorRT/ 放于此
│   ├── include/                    #   C++ 头文件（NvInfer.h 等）
│   ├── lib/                        #   动态库（libnvinfer.so 等）
│   └── bin/                        #   工具（trtexec 等）
│
└── demo介绍.ipynb                  # 本 notebook
```

> **说明**：`onnxruntime/` 和 `TensorRT/` 需要从官网下载后放到项目根目录下，
> CMakeLists.txt 中的路径已按此结构配置好，放到对应位置即可编译。

## 各文件说明

| 文件 | 作用 |
|------|------|
| `model/model.py` | 定义 SimpleMLP（3层MLP：4→16→16→3），提供 `get_model()` 和 `save_weights()` 接口 |
| `onnx/export_onnx.py` | 导出 ONNX（opset=17，支持动态 batch），做结构检查和 PyTorch vs ORT 数值对比 |
| `onnx/infer_python.py` | ONNX Runtime Python 推理：基本推理、动态 batch、性能优化配置、多次推理 |
| `onnx/cpp/main.cpp` | ONNX Runtime C++ 推理：Env→Session→CreateTensor→Run→取输出 |
| `tensorrt/build_engine.py` | 调用 `trtexec` 构建 FP32/FP16 Engine |
| `tensorrt/infer_python.py` | TensorRT Python 推理：加载 Engine→pycuda 分配显存→推理→取结果 |
| `tensorrt/cpp/main.cpp` | TensorRT C++ 推理：加载 Engine→cudaMalloc→executeV3→取结果 |

---
## 环境配置

### Python 依赖

| 包 | 用途 | 安装命令 |
|---|------|----------|
| torch | 模型定义与导出 | `pip install torch` |
| onnx | ONNX 结构检查 | `pip install onnx` |
| onnxruntime | ONNX Python 推理 | `pip install onnxruntime` |
| tensorrt | TensorRT Python 推理 | `pip install tensorrt` |
| pycuda | Python CUDA 内存管理 | `pip install pycuda` |

### C++ 依赖

**ONNX Runtime C++ 包**：
1. 从 [GitHub Releases](https://github.com/microsoft/onnxruntime/releases) 下载对应版本
2. 例如 `onnxruntime-linux-x64-1.19.2.tgz`
3. 解压到 `ONNX_TensorRT_demo/onnxruntime/`

```bash
cd ONNX_TensorRT_demo
wget https://github.com/microsoft/onnxruntime/releases/download/v1.19.2/onnxruntime-linux-x64-1.19.2.tgz
tar xzf onnxruntime-linux-x64-1.19.2.tgz
mv onnxruntime-linux-x64-1.19.2 onnxruntime
```

**TensorRT C++ 库**：
- 从 [NVIDIA Developer](https://developer.nvidia.com/tensorrt) 下载
- 安装后确保 `trtexec` 命令可用，`/usr/include/NvInfer.h` 存在

---
# 第一部分：ONNX 部署

## Step 1：导出 ONNX 模型

导出 ONNX 并自动验证结构和数值精度：

In [ ]:
# 查看导出脚本
!cat model/model.py

In [ ]:
# 运行导出
!python onnx/export_onnx.py

## Step 2：ONNX Runtime Python 推理

In [ ]:
!python onnx/infer_python.py

## Step 3：ONNX Runtime C++ 推理

### 3.1 下载 ONNX Runtime C++ 包（如果还没下载）

In [ ]:
# 取消注释来下载
# !cd .. && wget https://github.com/microsoft/onnxruntime/releases/download/v1.19.2/onnxruntime-linux-x64-1.19.2.tgz
# !cd .. && tar xzf onnxruntime-linux-x64-1.19.2.tgz && mv onnxruntime-linux-x64-1.19.2 onnxruntime

### 3.2 查看 C++ 代码

In [ ]:
!cat onnx/cpp/CMakeLists.txt

In [ ]:
!cat onnx/cpp/main.cpp

### 3.3 编译并运行

In [ ]:
!mkdir -p onnx/cpp/build && cd onnx/cpp/build && cmake .. && make

In [ ]:
!cd onnx/cpp/build && ./onnx_demo ../../model.onnx

---
# 第二部分：TensorRT 部署

> **注意**：TensorRT 需要先安装。如未安装，请参考上方「环境配置」部分。

## Step 4：构建 TensorRT Engine

In [ ]:
# 检查 trtexec 是否可用
!which trtexec 2>/dev/null && echo "✓ trtexec 已安装" || echo "✗ trtexec 未找到，请先安装 TensorRT"

In [ ]:
!python tensorrt/build_engine.py

## Step 5：TensorRT Python 推理

In [ ]:
!python tensorrt/infer_python.py

## Step 6：TensorRT C++ 推理

### 6.1 查看 C++ 代码

In [ ]:
!cat tensorrt/cpp/CMakeLists.txt

In [ ]:
!cat tensorrt/cpp/main.cpp

### 6.2 编译并运行

In [ ]:
!mkdir -p tensorrt/cpp/build && cd tensorrt/cpp/build && cmake .. && make

In [ ]:
!cd tensorrt/cpp/build && ./trt_demo ../../model.engine

---
## 关键 API 对照表

| 操作 | ONNX Runtime (Python) | ONNX Runtime (C++) | TensorRT (Python) | TensorRT (C++) |
|------|----------------------|-------------------|-------------------|----------------|
| 加载模型 | `ort.InferenceSession(path)` | `Ort::Session(env, path, opts)` | `runtime.deserialize_cuda_engine(bytes)` | `runtime->deserializeCudaEngine(data, size)` |
| 构造输入 | numpy array | `Ort::Value::CreateTensor()` | numpy + `cuda.memcpy_htod()` | `cudaMemcpy()` |
| 推理 | `session.run(None, {name: data})` | `session.Run(...)` | `context.execute_async_v3(stream)` | `context->enqueueV3(stream)` |
| 取输出 | `outputs[0]` (numpy) | `GetTensorMutableData<float>()` | `cuda.memcpy_dtoh()` | `cudaMemcpy()` |
| 关闭梯度 | 默认不需要 | 默认不需要 | 默认不需要 | 默认不需要 |

## ONNX vs TensorRT 选型

| 维度 | ONNX Runtime | TensorRT |
|------|-------------|----------|
| 硬件支持 | CPU + GPU + 多平台 | 仅 NVIDIA GPU |
| 部署难度 | 低（pip install 即可） | 中（需安装 TensorRT SDK） |
| GPU 推理速度 | 良好 | 极致（算子融合 + 低精度优化） |
| 精度模式 | FP32 / FP16 | FP32 / FP16 / INT8 |
| 动态 Shape | 原生支持 | 需指定 min/opt/max 范围 |
| 适用场景 | 通用部署、CPU 推理、跨平台 | GPU 推理性能要求高的场景 |

## 常见问题

**Q: ONNX 导出时报 Unsupported operator？**  
A: 提高 `opset_version`（如 17），或先用 `torch.jit.script` 转 TorchScript 再导出。

**Q: ONNX Runtime C++ 编译找不到头文件？**  
A: 确认已下载 ONNX Runtime 预编译包并解压到正确位置，检查 `CMakeLists.txt` 中的 `ORT_ROOT` 路径。

**Q: TensorRT trtexec 构建 Engine 失败？**  
A: 检查 ONNX 模型是否有不支持的算子，可用 `pip install onnxsim` + `onnxsim model.onnx model_sim.onnx` 简化模型。

**Q: TensorRT Python 推理报 CUDA 错误？**  
A: 确保 `pycuda` 已安装且 CUDA 驱动正常。`nvidia-smi` 能正常输出即可。

**Q: ONNX 和 TensorRT 输出数值有差异？**  
A: 正常现象。TensorRT 的算子融合和低精度优化会引入微小误差，一般在 1e-3 量级内可接受。